# T28 — Calibration with temperature scaling, all backbones

Inference only; nothing is trained.

For every checkpoint found, this notebook:
- re-computes test predictions and checks them against the results committed in the repo;
- fits one temperature on the **validation** split (never on test);
- reports ECE (15 bins), Brier score and NLL on the **test** split before and after scaling;
- saves reliability diagrams and the raw validation/test logits.

**Expected run time on one T4:** about 30–45 minutes for all 44 checkpoints
(setup and reading checkpoints about 5 min, caching the two splits about 3 min, then roughly
10–20 s per CNN checkpoint and 45–60 s per ViT-Base checkpoint).

**Before running (Kaggle sidebar):**
1. Session options → Accelerator → **GPU T4** (one is enough). Internet → **On**.
2. Add input → the primary dataset `tawsifurrahman/covid19-radiography-database`.
3. Add input → every dataset that holds the trained `.pt` checkpoints. The notebook searches all of
   `/kaggle/input` for `.pt` files and reads the backbone, arm and seed stored inside each one, so folder
   names do not matter. The seed-42 checkpoints are already in `uom230334h/t28-checkpoints`,
   `uom230334h/t28-vitbase-checkpoints` and `uom230334h/t28-efficientnet-checkpoints`. The seed 123 and
   2026 checkpoints (A0 and the selected arm of each backbone) still have to be uploaded as a dataset.
4. Run All. The inventory cell prints which checkpoints were found and which core-grid ones are missing;
   the run continues with whatever is present.

**Output:** `/kaggle/working/T28_calibration.zip`, containing `calibration_summary_all.csv`,
`checkpoint_verification.csv`, `calibration_core_3seed.csv` and one folder per checkpoint.

In [ ]:
import os, sys, json, time, math, hashlib, shutil, subprocess
from pathlib import Path

REPO_URL    = "https://github.com/Ravindu-Pathirana/Chest-X-ray-Disease-Detection.git"
REPO_BRANCH = "t26-vit-kaggle-results"   # holds the ViT-Base results; falls back to main if it has been merged and deleted

# Everything below is found automatically under /kaggle/input. The environment variables only exist
# so the notebook can also be dry-run off Kaggle.
DATA_ROOT = Path(os.environ.get("TRUST_DATA_ROOT", "/kaggle/input"))   # searched for COVID-19_Radiography_Dataset
CKPT_ROOT = Path(os.environ.get("TRUST_CKPT_ROOT", "/kaggle/input"))   # searched recursively for *.pt
WORK      = Path(os.environ.get("TRUST_WORK", "/kaggle/working"))
REPO      = Path(os.environ.get("TRUST_REPO", str(WORK / "Chest-X-ray-Disease-Detection")))
OUT       = WORK / "T28_calibration"

BATCH_SIZE  = 64
NUM_WORKERS = int(os.environ.get("TRUST_WORKERS", "4"))
DRY_RUN     = os.environ.get("TRUST_DRY_RUN", "0") == "1"   # set to True for a 2-checkpoint trial run

OUT.mkdir(parents=True, exist_ok=True)
print("output folder:", OUT, "| dry run:", DRY_RUN)

In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    if subprocess.run(["git", "-C", str(REPO), "checkout", "-q", REPO_BRANCH]).returncode != 0:
        print(f"branch {REPO_BRANCH} not found -> staying on the default branch")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repo commit:", COMMIT)

for module, package in [("timm", "timm"), ("pytorch_grad_cam", "grad-cam")]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if str(WORK).startswith("/kaggle") and DEVICE.type != "cuda":
    raise RuntimeError("No GPU. In the Kaggle sidebar set Session options -> Accelerator -> GPU T4, then Run All again.")

In [ ]:
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
SEEDS = [42, 123, 2026]
ARTIFACT = {"densenet121": "T18_lung_attention", "resnet50": "T23_resnet50_lung_attention",
            "efficientnet_b0": "T25_efficientnet_b0_lung_attention",
            "vit_base_patch16_224": "T26_vit_base_lung_attention"}
BASE_ARM = {"densenet121": "A0_vanilla", "resnet50": "A0_vanilla", "efficientnet_b0": "A0",
            "vit_base_patch16_224": "A0_vanilla"}
SELECTED_ARM = {"densenet121": "A2_full", "resnet50": "A3_multiply", "efficientnet_b0": "A3",
                "vit_base_patch16_224": "A3_multiply"}
SKIP_DIRS = {"images", "masks", ".venv", ".git", "__pycache__", "stage_2_train_images", "stage_2_test_images"}


def find_data_dir(root):
    for d, subdirs, _ in os.walk(root):
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
        p = Path(d)
        if (p / "COVID" / "images").is_dir() and (p / "Viral Pneumonia" / "masks").is_dir():
            return p
    raise FileNotFoundError(f"COVID-19_Radiography_Dataset (with images/ and masks/) not found under {root}. "
                            "Attach the dataset tawsifurrahman/covid19-radiography-database.")


def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)


def sha256_12(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()[:12]


def reference_dir(backbone, arm, seed):
    """Folder in the repo that holds this run's committed results."""
    base = REPO / "artifacts" / ARTIFACT[backbone]
    if backbone == "densenet121" and arm == "A2_bg":
        return base / "bg_suppression_followup" / "runs" / "A2_bg"
    if seed == 42:
        return base / "runs" / arm
    if backbone == "efficientnet_b0":
        return base / "runs_multiseed" / f"seed_{seed}" / "runs" / arm
    return base / "runs_multiseed" / f"{arm}_seed{seed}"


def discover_checkpoints(root):
    rows = []
    for d, subdirs, files in os.walk(root):
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
        for name in files:
            path = Path(d) / name
            if not name.endswith(".pt") or "smoke" in str(path).lower():
                continue
            ck = torch_load(path)
            if not (isinstance(ck, dict) and "model_state_dict" in ck and "config" in ck):
                print("skipped (not a project checkpoint):", path)
                continue
            seed = int(ck.get("seed", 42))
            arm = str(ck.get("arm", path.stem))
            if arm.endswith(f"_seed{seed}"):
                arm = arm[: -len(f"_seed{seed}")]
            model_cfg, module_cfg = ck["config"]["model"], ck["config"]["module"]
            backbone = model_cfg["name"]
            if backbone not in ARTIFACT:
                print("skipped (unknown backbone):", path)
                continue
            rows.append({
                "backbone": backbone, "arm": arm, "seed": seed,
                "use_attention": module_cfg.get("use_attention"), "attention": module_cfg.get("attention", "lung"),
                "gate_mode": module_cfg.get("gate_mode"), "lambda_att": module_cfg.get("lambda_att"),
                "lambda_bg": module_cfg.get("lambda_bg"),
                "size_mb": round(path.stat().st_size / 2**20, 1), "path": str(path),
            })
            del ck
    inv = pd.DataFrame(rows)
    if inv.empty:
        raise FileNotFoundError(f"no .pt checkpoints found under {root}. Attach the checkpoint datasets.")
    before = len(inv)
    inv = inv.sort_values(["backbone", "seed", "arm", "path"]).drop_duplicates(["backbone", "arm", "seed"])
    if len(inv) < before:
        print(f"{before - len(inv)} duplicate checkpoint files ignored (same backbone/arm/seed)")
    inv["role"] = [
        "baseline" if a == BASE_ARM[b] else "selected" if a == SELECTED_ARM[b] else "other"
        for b, a in zip(inv.backbone, inv.arm)]
    inv["run"] = inv.backbone + "/" + inv.arm + "/seed_" + inv.seed.astype(str)
    return inv.reset_index(drop=True)


DATA_DIR = find_data_dir(DATA_ROOT)
print("dataset:", DATA_DIR)
INVENTORY = discover_checkpoints(CKPT_ROOT)
print(f"{len(INVENTORY)} checkpoints found")
print(INVENTORY.groupby(["backbone", "seed"]).size().unstack(fill_value=0))

# What the core grid needs: baseline + selected arm at three seeds for every backbone (24 checkpoints).
have = set(zip(INVENTORY.backbone, INVENTORY.arm, INVENTORY.seed))
missing_core = [(b, a, s) for b in ARTIFACT for a in (BASE_ARM[b], SELECTED_ARM[b]) for s in SEEDS
                if (b, a, s) not in have]
print("core grid checkpoints missing:", len(missing_core))
for m in missing_core:
    print("   MISSING", m)

In [ ]:
from src.datasets import build_dataloaders
from src.modules import build_model


def load_model(row):
    """Rebuild the model from the settings stored inside the checkpoint, then load the weights strictly."""
    ck = torch_load(row["path"])
    model_cfg, module_cfg = ck["config"]["model"], ck["config"]["module"]
    if list(ck.get("class_names", CLASS_NAMES)) != CLASS_NAMES:
        raise ValueError(f"class order differs in {row['path']}")
    model = build_model(
        num_classes=model_cfg["num_classes"], use_attention=module_cfg["use_attention"],
        reduction=module_cfg.get("reduction", 8), gate_mode=module_cfg.get("gate_mode", "residual"),
        backbone_name=model_cfg["name"], pretrained=False,
        attention=module_cfg.get("attention", "lung"), drop_rate=model_cfg.get("drop_rate", 0.0))
    model.load_state_dict(ck["model_state_dict"], strict=True)
    return model.to(DEVICE).eval()


def free(model):
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


@torch.no_grad()
def predict_logits(model, X):
    """fp32 logits for a cached image tensor (no AMP: fp16 inputs flip about 2% of predictions)."""
    out = []
    for i in range(0, len(X), BATCH_SIZE):
        y = model(X[i:i + BATCH_SIZE].to(DEVICE))
        out.append((y[0] if isinstance(y, tuple) else y).float().cpu())
    return torch.cat(out)


def image_key(path):
    return "/".join(str(path).replace("\\", "/").split("/")[-3:])


def cache_split(loader, with_masks):
    xs, ys, ms = [], [], []
    for batch in loader:
        xs.append(batch[0]); ys.append(batch[1])
        if with_masks:
            ms.append(batch[2])
    return torch.cat(xs), torch.cat(ys), (torch.cat(ms) if with_masks else None)


def verify_against_committed(row, probs, labels, keys):
    """Compare this session's test predictions with the results committed in the repo."""
    ref = reference_dir(row["backbone"], row["arm"], row["seed"])
    pred = probs.argmax(1).numpy()
    acc = float((pred == labels.numpy()).mean())
    out = {"test_accuracy": acc, "committed_accuracy": None, "pred_agreement": None,
           "max_abs_prob_diff": None, "verified": "no committed reference"}
    js, cs = ref / "phase2_finetune_test_results.json", ref / "per_image_predictions.csv"
    if js.exists():
        cm = np.array(json.loads(js.read_text())["confusion_matrix"])
        out["committed_accuracy"] = float(np.trace(cm) / cm.sum())
        out["verified"] = "ok" if abs(acc - out["committed_accuracy"]) <= 1e-4 else "ACCURACY MISMATCH"
    if cs.exists():
        df = pd.read_csv(cs)
        df["key"] = df["image_path"].map(image_key)
        df = df.set_index("key").loc[keys]
        ref_pred = df["pred_label"].map({c: i for i, c in enumerate(CLASS_NAMES)}).to_numpy()
        ref_prob = df[[f"prob_{k}" for k in range(4)]].to_numpy()
        out["pred_agreement"] = float((ref_pred == pred).mean())
        out["max_abs_prob_diff"] = float(np.abs(ref_prob - probs.numpy()).max())
    return out


t0 = time.time()
_, val_loader, test_loader, class_names, _, datasets = build_dataloaders(
    DATA_DIR, img_size=224, batch_size=BATCH_SIZE, seed=42, num_workers=NUM_WORKERS,
    split_manifest_path="artifacts/splits/split_manifest_v1.csv")
assert list(class_names) == CLASS_NAMES, class_names
assert len(datasets["val"]) == 3175 and len(datasets["test"]) == 3175, "fixed split must be 3,175 val / 3,175 test"
TEST_DS = datasets["test"]
TEST_KEYS = [image_key(TEST_DS.base_dataset.samples[TEST_DS.indices[i]][0]) for i in range(len(TEST_DS))]
assert len(set(TEST_KEYS)) == len(TEST_KEYS)

X_val, y_val, _ = cache_split(val_loader, with_masks=False)
X_test, y_test, _ = cache_split(test_loader, with_masks=False)
print(f"cached val {tuple(X_val.shape)} and test {tuple(X_test.shape)} in {time.time() - t0:.0f}s")

In [ ]:
from src.modules import (brier_score, expected_calibration_error, fit_temperature,
                         plot_reliability_diagram, reliability_diagram_data)

N_BINS = 15
todo = INVENTORY.head(2) if DRY_RUN else INVENTORY
records = []
for n, row in enumerate(todo.to_dict("records"), 1):
    t1 = time.time()
    out_dir = OUT / row["backbone"] / row["arm"] / f"seed_{row['seed']}"
    done = out_dir / "calibration_summary.json"
    if done.exists():
        records.append(json.loads(done.read_text()))
        print(f"[{n}/{len(todo)}] {row['run']}: already done, skipped")
        continue
    out_dir.mkdir(parents=True, exist_ok=True)
    model = load_model(row)
    val_logits, test_logits = predict_logits(model, X_val), predict_logits(model, X_test)
    free(model)

    raw = F.softmax(test_logits, dim=1)
    check = verify_against_committed(row, raw, y_test, TEST_KEYS)
    temperature = fit_temperature(val_logits, y_val)
    scaled = F.softmax(test_logits / temperature, dim=1)

    rec = {
        "backbone": row["backbone"], "arm": row["arm"], "seed": row["seed"], "role": row["role"],
        "temperature": float(temperature),
        "ece_before": expected_calibration_error(raw, y_test, N_BINS),
        "ece_after": expected_calibration_error(scaled, y_test, N_BINS),
        "brier_before": brier_score(raw, y_test, len(CLASS_NAMES)),
        "brier_after": brier_score(scaled, y_test, len(CLASS_NAMES)),
        "nll_before": float(F.cross_entropy(test_logits, y_test)),
        "nll_after": float(F.cross_entropy(test_logits / temperature, y_test)),
        "mean_confidence_before": float(raw.max(1).values.mean()),
        "mean_confidence_after": float(scaled.max(1).values.mean()),
        "val_accuracy": float((val_logits.argmax(1) == y_val).float().mean()),
        **check, "n_bins": N_BINS, "test_n": len(y_test), "val_n": len(y_val),
        "checkpoint": row["path"], "checkpoint_sha256_12": sha256_12(row["path"]), "repo_commit": COMMIT,
    }
    name = f"{row['backbone']} {row['arm']} seed {row['seed']}"
    plot_reliability_diagram(reliability_diagram_data(raw, y_test, N_BINS),
                             f"{name} - before (T=1.0, ECE={rec['ece_before']:.4f})", out_dir / "reliability_before.png")
    plot_reliability_diagram(reliability_diagram_data(scaled, y_test, N_BINS),
                             f"{name} - after (T={temperature:.3f}, ECE={rec['ece_after']:.4f})", out_dir / "reliability_after.png")
    np.savez_compressed(out_dir / "logits.npz", val_logits=val_logits.numpy(), val_labels=y_val.numpy(),
                        test_logits=test_logits.numpy(), test_labels=y_test.numpy(), test_keys=np.array(TEST_KEYS))
    done.write_text(json.dumps(rec, indent=2))
    records.append(rec)
    print(f"[{n}/{len(todo)}] {row['run']}: acc {rec['test_accuracy']:.4f} ({rec['verified']})  T={temperature:.3f}  "
          f"ECE {rec['ece_before']:.4f} -> {rec['ece_after']:.4f}  ({time.time() - t1:.0f}s)")

In [ ]:
summary = pd.DataFrame(records).sort_values(["backbone", "seed", "arm"]).reset_index(drop=True)
summary.to_csv(OUT / "calibration_summary_all.csv", index=False)
summary[["backbone", "arm", "seed", "test_accuracy", "committed_accuracy", "pred_agreement", "max_abs_prob_diff",
         "verified", "checkpoint_sha256_12", "checkpoint"]].to_csv(OUT / "checkpoint_verification.csv", index=False)

bad = summary[summary.verified == "ACCURACY MISMATCH"]
print(f"{len(summary)} checkpoints calibrated | verified ok: {(summary.verified == 'ok').sum()} | "
      f"mismatch: {len(bad)} | no reference: {(summary.verified == 'no committed reference').sum()}")
if len(bad):
    print("DO NOT USE these rows - the checkpoint does not reproduce the committed accuracy:")
    print(bad[["backbone", "arm", "seed", "test_accuracy", "committed_accuracy"]])
print("ECE got worse after scaling for:", int((summary.ece_after > summary.ece_before).sum()), "checkpoints")

pd.set_option("display.width", 200)
print(summary[["backbone", "arm", "seed", "temperature", "ece_before", "ece_after", "brier_before", "brier_after",
               "nll_before", "nll_after"]].round(4).to_string(index=False))

In [ ]:
# Baseline vs selected arm, mean over the seeds available (the comparison the paper reports).
core = summary[summary.role.isin(["baseline", "selected"])]
metrics = ["temperature", "ece_before", "ece_after", "brier_before", "brier_after", "nll_before", "nll_after"]
core_table = core.groupby(["backbone", "role", "arm"])[metrics].agg(["mean", "std"]).round(4)
core_table.insert(0, "n_seeds", core.groupby(["backbone", "role", "arm"]).size())
core_table.to_csv(OUT / "calibration_core_3seed.csv")
print(core_table.to_string())

In [ ]:
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Download this file from the Output panel:", archive, f"({Path(archive).stat().st_size / 2**20:.1f} MB)")